## Importación de librerías

En esta sección se importan las bibliotecas necesarias para el desarrollo de la actividad.

- **sqlite3:** permite crear conexiones y gestionar bases de datos SQLite desde Python.
- **pandas:** proporciona estructuras de datos y herramientas para la carga, limpieza, transformación y análisis de datos tabulares.

Estas librerías serán utilizadas para leer el conjunto de datos, realizar el procesamiento requerido y almacenar los resultados en una base de datos SQLite para su posterior consulta.

In [9]:
import sqlite3
import pandas as pd

## Carga del dataset y creación de la base de datos SQLite

En esta etapa se realiza la lectura del archivo **ventas_techventas.csv** utilizando la librería Pandas. Posteriormente, se establece una conexión con una base de datos SQLite llamada **techventas.db**.

Finalmente, el conjunto de datos es almacenado en una tabla denominada **ventas**, reemplazando cualquier versión anterior existente. Este proceso permite persistir la información en una base de datos relacional para facilitar consultas posteriores y análisis mediante SQL.

### Acciones realizadas

1. Lectura del archivo CSV.
2. Creación de la conexión a SQLite.
3. Creación o reemplazo de la tabla **ventas**.
4. Almacenamiento de los registros en la base de datos.

In [10]:
df = pd.read_csv("ventas_techventas.csv")
conn = sqlite3.connect("techventas.db")
df.to_sql("ventas", conn, if_exists="replace", index=False)

60

## Consulta Q1 - Productos únicos disponibles

Con el objetivo de identificar el catálogo de productos registrados en la base de datos, se utiliza la cláusula **SELECT DISTINCT**, la cual permite obtener únicamente valores únicos sin repeticiones.

Esta consulta facilita la exploración inicial de los datos y permite conocer los diferentes productos comercializados por la empresa.

### Objetivo

- Identificar los productos disponibles.
- Eliminar registros duplicados.
- Obtener una visión general del catálogo de ventas.

### Consulta SQL utilizada

```sql
SELECT DISTINCT producto AS producto_unico
FROM ventas;
```

In [11]:
pd.read_sql_query("""
SELECT DISTINCT producto AS producto_unico
FROM ventas;
""", conn)

,producto_unico
0,Laptop Pro 15
1,Mouse Inalambrico
2,NaN
3,Teclado Mecanico
4,SSD 1TB
5,Router WiFi 6


## Consulta Q2 - Pedidos del primer trimestre con cantidad mayor a 5 unidades

Esta consulta utiliza las cláusulas **WHERE** y **BETWEEN** para filtrar los registros correspondientes al primer trimestre del año 2024 (enero, febrero y marzo). Además, se seleccionan únicamente aquellos pedidos cuya cantidad vendida sea superior a cinco unidades.

### Objetivo

- Filtrar ventas realizadas durante el primer trimestre de 2024.
- Identificar pedidos con volúmenes de venta significativos.
- Analizar transacciones con cantidades superiores a cinco unidades.

### Consulta SQL utilizada

```sql
SELECT *
FROM ventas
WHERE fecha BETWEEN '2024-01-01' AND '2024-03-31'
AND cantidad > 5;
```

In [12]:
pd.read_sql_query("""
SELECT *
FROM ventas
WHERE fecha BETWEEN '2024-01-01' AND '2024-03-31'
AND cantidad > 5;
""", conn)

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15.0,25.0,0.00,Carlos Ruiz
1,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10.0,85.0,0.10,Luis Mora
2,1006,2024-01-18,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,20.0,95.0,0.00,Maria Torres
3,1008,2024-01-22,C006,NetPrime,Norte,Router WiFi 6,Redes,8.0,120.0,0.00,Luis Mora
4,1009,2024-01-25,C003,DataSolutions,Centro,SSD 1TB,Almacenamiento,12.0,95.0,0.10,Maria Torres
5,1012,2024-02-05,C008,BetaSoft,Centro,Mouse Inalambrico,Perifericos,30.0,25.0,0.05,Maria Torres
6,1015,2024-02-12,C005,Sistemas Globales,Oeste,SSD 1TB,Almacenamiento,25.0,95.0,0.10,Maria Torres
7,1018,2024-02-20,C002,Innovatek,Sur,SSD 1TB,Almacenamiento,18.0,95.0,0.00,Carlos Ruiz
8,1019,2024-02-22,C007,AlphaTech,Sur,Mouse Inalambrico,Perifericos,12.0,25.0,0.00,Maria Torres
9,1022,2024-03-04,C001,TechCorp SA,Norte,SSD 1TB,Almacenamiento,30.0,95.0,0.15,Carlos Ruiz


## Consulta Q3 - Ingreso bruto por vendedor

Esta consulta utiliza las cláusulas **GROUP BY** y **HAVING** para calcular el ingreso bruto generado por cada vendedor y filtrar únicamente aquellos cuyo total de ventas supera los 10.000 unidades monetarias.

El ingreso bruto se obtiene multiplicando la cantidad vendida por el precio unitario de cada producto y posteriormente sumando estos valores para cada vendedor.

### Objetivo

- Calcular el ingreso bruto generado por cada vendedor.
- Agrupar las ventas por responsable comercial.
- Identificar los vendedores con mejor desempeño.
- Filtrar únicamente aquellos que superan los 10.000 en ingresos.

### Consulta SQL utilizada

```sql
SELECT
    vendedor,
    SUM(cantidad * precio_unitario) AS ingreso_bruto
FROM ventas
GROUP BY vendedor
HAVING SUM(cantidad * precio_unitario) > 10000;
```

In [13]:
pd.read_sql_query("""
SELECT
    vendedor,
    SUM(cantidad * precio_unitario) AS ingreso_bruto
FROM ventas
GROUP BY vendedor
HAVING SUM(cantidad * precio_unitario) > 10000;
""", conn)

,vendedor,ingreso_bruto
0,Ana L��,20810.0
1,Carlos Ruiz,21355.0
2,Luis Mora,19830.0
3,Maria Torres,11860.0


## Consulta Q4 - Estadísticas de ventas por categoría

Esta consulta permite obtener un resumen estadístico de las ventas agrupadas por categoría de producto. Para ello se utilizan funciones de agregación que calculan el número total de pedidos, la cantidad de unidades vendidas y el precio promedio de los productos registrados en cada categoría.

El análisis por categorías facilita la comprensión del comportamiento comercial de los diferentes grupos de productos y permite identificar tendencias de ventas dentro del negocio.

### Objetivo

- Contabilizar la cantidad de pedidos por categoría.
- Calcular el total de unidades vendidas.
- Obtener el precio promedio de los productos.
- Analizar el desempeño comercial de cada categoría.

### Consulta SQL utilizada

```sql
SELECT
    categoria,
    COUNT(*) AS total_pedidos,
    SUM(cantidad) AS unidades_vendidas,
    AVG(precio_unitario) AS precio_promedio
FROM ventas
GROUP BY categoria;
```

In [14]:
pd.read_sql_query("""
SELECT
    categoria,
    COUNT(*) AS total_pedidos,
    SUM(cantidad) AS unidades_vendidas,
    AVG(precio_unitario) AS precio_promedio
FROM ventas
GROUP BY categoria;
""", conn)

,categoria,total_pedidos,unidades_vendidas,precio_promedio
0,NaN,11,NaN,NaN
1,Almacenamiento,12,260.0,95.000000
2,Electronica,14,31.0,1139.285714
3,Perifericos,15,215.0,53.000000
4,Redes,8,39.0,120.000000


## Creación de la tabla de vendedores

Con el objetivo de complementar la información contenida en la tabla **ventas**, se creó una tabla auxiliar denominada **vendedores**. Esta tabla almacena información relacionada con la zona de trabajo y la meta mensual asignada a cada vendedor.

La utilización de esta tabla permitirá posteriormente realizar consultas con la cláusula **JOIN**, combinando la información de ventas con los objetivos comerciales establecidos para cada vendedor.

### Acciones realizadas

1. Creación de la tabla **vendedores** si no existe previamente.
2. Eliminación de registros anteriores para evitar duplicidad de datos.
3. Inserción de información correspondiente a los vendedores.
4. Confirmación de los cambios mediante la instrucción `commit()`.

### Estructura de la tabla

| Campo | Tipo de dato | Descripción |
|---------|-------------|-------------|
| vendedor | TEXT | Nombre del vendedor |
| zona | TEXT | Zona geográfica asignada |
| meta_mensual | REAL | Meta de ventas mensual |

### Datos cargados

| Vendedor | Zona | Meta mensual |
|-----------|--------|-------------|
| Ana López | Norte | 8000 |
| Carlos Ruiz | Sur | 7500 |
| Luis Mora | Norte | 8000 |
| María Torres | Centro | 7000 |

Esta tabla servirá como base para calcular indicadores de desempeño y cumplimiento de metas comerciales mediante consultas SQL que integren información de múltiples tablas.

In [15]:
conn.executescript("""
CREATE TABLE IF NOT EXISTS vendedores(
    vendedor TEXT,
    zona TEXT,
    meta_mensual REAL
);

DELETE FROM vendedores;

INSERT INTO vendedores VALUES
('Ana López','Norte',8000),
('Carlos Ruiz','Sur',7500),
('Luis Mora','Norte',8000),
('María Torres','Centro',7000);
""")

conn.commit()

In [16]:
pd.read_sql_query("""
SELECT
    v.vendedor,
    vd.zona,
    vd.meta_mensual,
    SUM(v.cantidad * v.precio_unitario) AS ventas_totales,
    ROUND(
        (SUM(v.cantidad * v.precio_unitario) /
        vd.meta_mensual) * 100,
        2
    ) AS porcentaje_cumplimiento
FROM ventas v
JOIN vendedores vd
ON v.vendedor = vd.vendedor
GROUP BY
    v.vendedor,
    vd.zona,
    vd.meta_mensual;
""", conn)

,vendedor,zona,meta_mensual,ventas_totales,porcentaje_cumplimiento
0,Carlos Ruiz,Sur,7500.0,21355.0,284.73
1,Luis Mora,Norte,8000.0,19830.0,247.87


## Consulta Q6 - Cliente con mayor ingreso generado durante el primer semestre de 2024

Esta consulta utiliza una **subconsulta anidada** para identificar al cliente que generó el mayor ingreso total durante el primer semestre del año 2024.

El ingreso total se calcula multiplicando la cantidad vendida por el precio unitario de cada producto. Posteriormente, los resultados son agrupados por cliente y comparados con el valor máximo obtenido mediante una subconsulta.

Este tipo de consulta resulta útil cuando se desea identificar registros destacados dentro de un conjunto de datos, como el mejor cliente, el producto más vendido o el vendedor con mayores ingresos.

### Objetivo

- Calcular el ingreso total generado por cada cliente.
- Analizar las ventas realizadas durante el primer semestre de 2024.
- Identificar el cliente con mayor contribución económica.
- Aplicar subconsultas para obtener valores máximos dentro de un conjunto de resultados.

### Consulta SQL utilizada

```sql
SELECT
    cliente_nombre,
    SUM(cantidad * precio_unitario) AS ingreso_total
FROM ventas
WHERE fecha BETWEEN '2024-01-01' AND '2024-06-30'
GROUP BY cliente_nombre
HAVING ingreso_total = (
    SELECT MAX(total_ingreso)
    FROM (
        SELECT
            SUM(cantidad * precio_unitario) AS total_ingreso
        FROM ventas
        WHERE fecha BETWEEN '2024-01-01' AND '2024-06-30'
        GROUP BY cliente_nombre
    )
);
```

In [17]:
pd.read_sql_query("""
SELECT
    cliente_nombre,
    SUM(cantidad * precio_unitario) AS ingreso_total
FROM ventas
WHERE fecha BETWEEN '2024-01-01' AND '2024-06-30'
GROUP BY cliente_nombre
HAVING ingreso_total = (
    SELECT MAX(total_ingreso)
    FROM (
        SELECT
            SUM(cantidad * precio_unitario) AS total_ingreso
        FROM ventas
        WHERE fecha BETWEEN '2024-01-01' AND '2024-06-30'
        GROUP BY cliente_nombre
    )
);
""", conn)

,cliente_nombre,ingreso_total
0,TechCorp SA,19025.0
